In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
%matplotlib inline
import os

from v1sh_model.inputs.visualize import visualize_input, visualize_output
from v1sh_model.utils.connections import (
    compute_connection_kernel,
    visualize_weights,
)

In [ ]:
NI, NJ, NC = 35, 36, 79
K = 12
DI, DJ = 17, 18

# Load the C-generated 4D array
cpp_flat = np.loadtxt("Cpp.csv", delimiter=",")
J_original = cpp_flat.reshape((NI, NJ, K, K), order='C') # N_x, N_y , K_pre, K_post
J_original = np.swapaxes(J_original, 0, 1)
J_original = J_original[DJ-10:DJ+10+1, DI-10:DI+10+1, :, :]

cpm_flat = np.loadtxt("Cpm.csv", delimiter=",")
W_original = cpm_flat.reshape((NI, NJ, K, K), order='C')
W_original = np.swapaxes(W_original, 0, 1)
W_original = W_original[DJ-10:DJ+10+1, DI-10:DI+10+1, :, :]

# mapping in Zhaoping's code
# ang_original = np.zeros(K)
# ang_me = np.zeros(K)
# for k in range(K):
#     ang_original[k] = np.pi / 2.0 - k * np.pi / K
#     ang_me[k] = k * np.pi / K


In [ ]:
J, W, Psi = compute_connection_kernel(K=12, verbose=False, use_original=False)

In [ ]:
print("Original sums:")
print(J_original.sum(), W_original.sum())

print("My sums:")
print(J.sum(), W.sum())

In [ ]:
n_prec_diff_W = np.isclose(W_original, W, atol = 1e-4).sum() / W_original.size
n_prec_diff_J = np.isclose(J_original, J, atol = 1e-4).sum() / J_original.size
print(f"Normalized percent difference W: {n_prec_diff_W * 100}%")
print(f"Normalized percent difference J: {n_prec_diff_J * 100}%")

In [ ]:
plt.figure(figsize=(6, 4), dpi=400)
W_flat, W_ori_flat = W.flatten(), W_original.flatten()
plt.hist(W_flat[W_flat != 0.], range=(0, 0.14), bins=50, alpha=0.5, label='replicated')
plt.hist(W_ori_flat[W_ori_flat != 0.], range=(0, 0.14), bins=200, alpha=0.5, label='original')
plt.legend()
plt.title('$W$ Distribution')
plt.show()

plt.figure(figsize=(6, 4), dpi=400)
J_flat, J_ori_flat = J.flatten(), J_original.flatten()
plt.hist(J_flat[J_flat != 0], bins=200, range=(0, 0.13), alpha=0.5, label='replicated')
plt.hist(J_ori_flat[J_ori_flat != 0.], range=(0, 0.13), bins=200, alpha=0.5, label='original')
plt.legend()
plt.title('$J$ Distribution')
plt.show()

In [ ]:
for k in [0, 7]:
    fig1, fig2 = visualize_weights(W_original, J_original, K=K, k_pre=k, dpi=500, colored=False)
    print(f"Orientation = {k * 180 / K}°")
    # fig1.suptitle("J (Original)\n")
    # fig2.suptitle("W (original)\n")
    plt.show()
    plt.show()
    fig1, fig2 = visualize_weights(W, J, K=K, k_pre=k, dpi=500, colored=False)
    print(f"Orientation = {k * 180 / K}°")
    # fig1.suptitle("J\n")
    # fig2.suptitle("W\n")
    plt.show()
    plt.show()

In [ ]:
# save original connections as npy files 
np.save("J_original.npy", J_original)
np.save("W_original.npy", W_original)